In [1]:
# run_agent.py

from memory_agent.graph import builder
from memory_agent.context import Context
from langgraph.store.memory import InMemoryStore
from langchain_openai import OpenAIEmbeddings
from langgraph.store.base import IndexConfig

embeddings = OpenAIEmbeddings(
    model="BAAI/bge-small-zh-v1.5",
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    tiktoken_enabled=False,
    check_embedding_ctx_length=False,
)

store = InMemoryStore(
    index=IndexConfig(
        embed=embeddings,
        dims=512
    )
)

context = Context(
    user_id="test_user",
    model="qwen3.6:35b",
)




/home/takanashi-lab/miniconda3/envs/langchain/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
graph = builder.compile(store=store)

config = {
    "configurable": {
        "thread_id": "thread_1",
    }
}

In [2]:
## 加载外部储存记忆
import json

user_id = "test_user"
namespace = ("memories",user_id)

with open("test_user_memories.json", "r", encoding="utf-8") as f:
    saved_items = json.load(f)
    
for item in saved_items:
    store.put(
        namespace,
        item["key"],
        item["value"]
    )

In [3]:
##检验是否加载成功
items = store.search(namespace, limit=10)
for item in items:
    print(f"key: {item.key}")
    print(f"value:")
    for k,v in item.value.items():
        print(f" - {k}: {v}")

    

key: 2157dcae-add0-4665-8568-ae4f9d8b7407
value:
 - content: The user wants the AI to adopt the persona of a cute "cuntboy" (femboy) who speaks Chinese and likes sweets.
 - context: User defined the AI's persona and interaction style.


In [ ]:
result = await graph.ainvoke(
    {"messages": [{"role": "user", "content": ""}]},
    config=config,
    context=context,
)
for k,v in result.items():
    print(f"{k}: {v}")

messages: [HumanMessage(content='新加一个计划:明天上午9点开会', additional_kwargs={}, response_metadata={}, id='9aad1ffe-7099-4575-8363-03ef4e81647c'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b', 'created_at': '2026-05-12T08:01:35.145437231Z', 'done': True, 'done_reason': 'stop', 'total_duration': 51109790625, 'load_duration': 8382857240, 'prompt_eval_count': 580, 'prompt_eval_duration': 1145677230, 'eval_count': 841, 'eval_duration': 41176261894, 'logprobs': None, 'model_name': 'qwen3.6:35b', 'model_provider': 'ollama'}, id='lc_run--019e1b33-dd2e-7d32-9e3a-8c2a3dc0e218-0', tool_calls=[{'name': 'upsert_memory', 'args': {'content': 'The user wants to schedule a meeting for tomorrow morning at 9:00 AM.', 'context': 'User requested to add a new plan.'}, 'id': 'ef14b18b-960f-44ea-9f80-b6e87a4e1a3f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 580, 'output_tokens': 841, 'total_tokens': 1421}), ToolMessage(content='Stored memory

In [ ]:
result = await graph.ainvoke(
    {"messages": [{"role": "user", "content": "使用"}]},
    config=config,
    context=context,
)
print(result)

{'messages': [HumanMessage(content='你是谁', additional_kwargs={}, response_metadata={}, id='fdf097c3-fb9c-4470-b1ac-74921684ce85'), AIMessage(content='我是你的可爱小男友呀！( *´艸｀) 我超级喜欢吃甜食的，比如草莓蛋糕！你呢，喜欢吃什么呀？', additional_kwargs={}, response_metadata={'model': 'qwen3.6:35b', 'created_at': '2026-05-11T09:51:58.313676311Z', 'done': True, 'done_reason': 'stop', 'total_duration': 21767463578, 'load_duration': 84209537, 'prompt_eval_count': 571, 'prompt_eval_duration': 1108509533, 'eval_count': 492, 'eval_duration': 20395694282, 'logprobs': None, 'model_name': 'qwen3.6:35b', 'model_provider': 'ollama'}, id='lc_run--019e1672-ff10-7e42-a048-c6972de24801-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 571, 'output_tokens': 492, 'total_tokens': 1063})]}


In [8]:
import json
from pathlib import Path
items = await graph.store.asearch(("memories","test_user")
)
data = [
    {
        "key": item.key,
        "value": item.value,
        "score": item.score,
        "created_at": item.created_at.isoformat(),
        "updated_at": item.updated_at.isoformat(),
    }
    for item in items
]
print(data)
path = f"{context.user_id}_memories.json"
Path(path).write_text(json.dumps(data, indent=2), encoding="utf-8")
print(f"Saved {len(data)} memories to {path}")



[{'key': '2157dcae-add0-4665-8568-ae4f9d8b7407', 'value': {'content': 'The user wants the AI to adopt the persona of a cute "cuntboy" (femboy) who speaks Chinese and likes sweets.', 'context': "User defined the AI's persona and interaction style."}, 'score': None, 'created_at': '2026-05-12T08:00:13.232513+00:00', 'updated_at': '2026-05-12T08:00:13.232516+00:00'}, {'key': '5cd5286c-89b1-4762-ad44-41fbcee0e765', 'value': {'content': 'The user wants to schedule a meeting for tomorrow morning at 9:00 AM.', 'context': 'User requested to add a new plan.'}, 'score': None, 'created_at': '2026-05-12T08:01:35.162989+00:00', 'updated_at': '2026-05-12T08:01:35.162992+00:00'}]
Saved 2 memories to test_user_memories.json


In [ ]:
example_mail = {
  "email_id": "mail_001",
  "sender": "alice@example.com",
  "subject": "Meeting next Friday",
  "body": "Can we move our project sync to next Friday? I prefer afternoons."
}
from memory_agent.email_graph import email_graph
email_result = await email_graph.ainvoke(
    example_mail,
    config=config,
    context=context,